In [2]:
import numpy as np

def local_matrices_general2(x1, x2, Nve):
    """
    Calcule (A,b) pour un élément 1D P1 ou P2 Lagrange.

    Paramètres :
        x1, x2 : bornes de l'élément
        Nve    : 1 => P1 (2 nœuds), 2 => P2 (3 nœuds)

    Retour :
        A : matrice locale
        b : vecteur local
    """

    h = x2 - x1

    # -------------------------
    # CAS P1 (2 nœuds)
    # -------------------------
    if Nve == 1:
        # Matrice de rigidité
        A_deriv = (1/h) * np.array([[1, -1],
                                    [-1, 1]])

        # Matrice de masse
        A_mass = (h/6) * np.array([[2, 1],
                                   [1, 2]])

        # Matrice totale
        A = A_deriv + A_mass

        # Second membre
        b1 = (h/6) * (2*x1 + x2)
        b2 = (h/6) * (x1 + 2*x2)
        b = np.array([b1, b2])

        return A, b


    # -------------------------
    # CAS P2 (3 nœuds)
    # -------------------------
    elif Nve == 2:
        # Matrice de rigidité P2 standard (sur [-1,1])
        A_deriv_ref = np.array([
            [ 7/3, -8/3,  1/3],
            [-8/3, 16/3, -8/3],
            [ 1/3, -8/3,  7/3]
        ]) * (1/h)

        # Matrice de masse P2 standard
        A_mass_ref = np.array([
            [4, 2, -1],
            [2, 16, 2],
            [-1, 2, 4]
        ]) * (h/30)

        A = A_deriv_ref + A_mass_ref

        # Second membre b = ∫ x φ_i
        # Noeuds P2 : gauche, milieu, droite
        xm = (x1 + x2) / 2

        phi_nodes = np.array([x1, xm, x2])

        # Formule exacte de ∫ x φ_i sur un P2 :
        b = np.zeros(3)
        b[0] = h/30 * ( 6*x1 + 4*xm - x2 )
        b[1] = h/30 * ( 4*x1 +16*xm + 4*x2 )
        b[2] = h/30 * ( -x1 + 4*xm + 6*x2 )

        return A, b

    else:
        raise ValueError("Nve doit être 1 (P1) ou 2 (P2)")

print(local_matrices_general2(2,3,1))
print(local_matrices_general2(2,3,2))

(array([[ 1.33333333, -0.83333333],
       [-0.83333333,  1.33333333]]), array([1.16666667, 1.33333333]))
(array([[ 2.46666667, -2.6       ,  0.3       ],
       [-2.6       ,  5.86666667, -2.6       ],
       [ 0.3       , -2.6       ,  2.46666667]]), array([0.63333333, 2.        , 0.86666667]))


In [3]:
import numpy as np

# ------------------------------------------------------------
# Aire du triangle
# ------------------------------------------------------------
def area(x1, x2, x3):
    return 0.5 * abs(np.linalg.det(np.array([
        [x2[0]-x1[0], x3[0]-x1[0]],
        [x2[1]-x1[1], x3[1]-x1[1]]
    ])))

# ------------------------------------------------------------
# Gradients des fonctions de forme P1 (constants)
# ------------------------------------------------------------
def grad_lambda(x1, x2, x3):
    A = area(x1, x2, x3)
    b = np.array([x2[1] - x3[1], x3[1] - x1[1], x1[1] - x2[1]])
    c = np.array([x3[0] - x2[0], x1[0] - x3[0], x2[0] - x1[0]])
    G = np.zeros((3,2))
    for i in range(3):
        G[i,:] = np.array([b[i], c[i]]) / (2 * A)
    return G, A

# ------------------------------------------------------------
# Élément (matrice A et vecteur B) pour -Mu Δu + αu = f
# ------------------------------------------------------------
def elem_matrix_vector(x1, x2, x3, Mu, alpha, rhs_fun):
    G, A = grad_lambda(x1, x2, x3)

    Ae = Mu * A * (G @ G.T)

    # Masse (réaction)
    Me = alpha * A / 12 * np.array([
        [2,1,1],
        [1,2,1],
        [1,1,2]
    ])

    Ae += Me

    # Second membre : quadrature au centre du triangle
    xc = (x1 + x2 + x3) / 3.0
    f = rhs_fun(xc)

    Be = f * A / 3.0 * np.ones((3,1))

    return Ae, Be

# ------------------------------------------------------------
# Test
# ------------------------------------------------------------
def rhs(x):  # exemple
    return 1.0

x1 = np.array([0.0,0.0])
x2 = np.array([2.0,0.0])
x3 = np.array([0.0,2.0])

Mu = 1.0
alpha = 1.0

Ae, Be = elem_matrix_vector(x1, x2, x3, Mu, alpha, rhs)

print("Matrice élémentaire Ae :")
print(Ae)
print("\nVecteur élémentaire Be :")
print(Be)


Matrice élémentaire Ae :
[[ 1.33333333 -0.33333333 -0.33333333]
 [-0.33333333  0.83333333  0.16666667]
 [-0.33333333  0.16666667  0.83333333]]

Vecteur élémentaire Be :
[[0.66666667]
 [0.66666667]
 [0.66666667]]
